# Appendix D: Autocorrelation of the Squared Envelope

This notebook derives the closed-form autocorrelation
$R_\Gamma(\tau) = \int_{-\infty}^{\infty} \Gamma(s)\,\Gamma(s+\tau)\,ds$
of the squared trapezoidal envelope $\Gamma(t) = \alpha^2(t)$.

**Method:** Since $\Gamma$ is a piecewise polynomial with compact support,
split the convolution integral into sub-intervals where both $\Gamma(s)$
and $\Gamma(s+\tau)$ have known polynomial forms.

**Key result:** $R_\Gamma(\tau)$ is a piecewise degree-5 polynomial on
four intervals: $[0,\tau_s]$, $[\tau_s,\ell]$, $[\ell,\ell+\tau_s]$,
$[\ell+\tau_s,\ell+2\tau_s]$, and zero beyond.

In [ ]:
import sympy as sp
from sympy import pi, cos, sin, integrate, simplify, symbols, latex, sqrt, Rational, Poly
from IPython.display import display, Math
sp.init_printing()

## Setup

$\Gamma(t)$ has three regions on $[-S, S]$ where $S = \ell/2 + \tau_s$:
- **Ramp (negative):** $s \in [-S, -L]$: $\Gamma(s) = \alpha_{\max}^2\,((s+S)/\tau_s)^2$
- **Plateau:** $s \in [-L, L]$: $\Gamma(s) = \alpha_{\max}^2$
- **Ramp (positive):** $s \in [L, S]$: $\Gamma(s) = \alpha_{\max}^2\,((S-s)/\tau_s)^2$

where $L = \ell/2$. The derivation below assumes $L > \tau_s$, i.e. $\ell > 2\tau_s$, which fixes the order of the sub-interval breakpoints. The paper states the result for all $\ell \geq \tau_s$; the section *Validity range* at the end checks that, and that it fails for $\ell < \tau_s$.

In [ ]:
s = sp.Symbol('s', real=True)
T = sp.Symbol('T', positive=True)  # lag variable
L = sp.Symbol('L', positive=True)  # ell/2
ts = sp.Symbol('tau_s', positive=True)
amax = sp.Symbol('alpha_max', positive=True)

a2 = amax**2  # shorthand
S = L + ts    # half-support

# Gamma expressions in each region
ramp_neg_s = a2 * ((s + S) / ts)**2       # s in [-S, -L]
plateau_s  = a2                             # s in [-L, L]
ramp_pos_s = a2 * ((S - s) / ts)**2        # s in [L, S]

# Gamma(s+T) expressions
ramp_neg_st = a2 * ((s + T + S) / ts)**2   # s+T in [-S, -L]
plateau_st  = a2                             # s+T in [-L, L]
ramp_pos_st = a2 * ((S - s - T) / ts)**2   # s+T in [L, S]

### Check: input envelope matches the paper (Eq. `eq:Gamma_piecewise`)

The paper defines the **normalized** squared envelope $\Gamma(t) = \alpha^2(t)/\alpha_{\max}^2$ in Eq. `eq:Gamma_piecewise`, which is
taken from the equation registry (`equations.py`), where it is transcribed exactly as typeset. In that equation $\tau$ means
$\tau_{\rm spot}$, not the lag. This notebook keeps the factor $\alpha_{\max}^2$, so we check that each region divided by
$\alpha_{\max}^2$ equals the paper's expression with $\ell = 2L$, and that the shifted pieces really are $\Gamma(s+T)$.

In [ ]:
# Eq. (Gamma_piecewise) from the equation registry (there: ell = ell_spot, tau = tau_spot, which is ts here)
import equations as eqs

Gamma_paper = eqs.get("eq:Gamma_piecewise").expr.subs(eqs.ell, 2 * L)
t_abs = sp.Symbol('t_abs', positive=True)  # |t|
branches = [e.subs(sp.Abs(eqs.t), t_abs) for e, _ in Gamma_paper.args]
assert len(branches) == 3 and branches[0] == 0                # zero beyond the support
Gamma_paper_ramp, Gamma_paper_plateau = branches[1], branches[2]  # ell/2 < |t| <= ell/2 + tau, |t| <= ell/2

# Gamma(s) / alpha_max^2 vs paper: negative ramp has |t| = -s, positive ramp has |t| = s
assert simplify(ramp_neg_s / a2 - Gamma_paper_ramp.subs(t_abs, -s)) == 0
assert simplify(ramp_pos_s / a2 - Gamma_paper_ramp.subs(t_abs, s)) == 0
assert simplify(plateau_s / a2 - Gamma_paper_plateau) == 0
# ramp meets the plateau (Gamma = 1 at |t| = ell/2) and vanishes at |t| = ell/2 + tau
assert simplify(Gamma_paper_ramp.subs(t_abs, L) - 1) == 0
assert simplify(Gamma_paper_ramp.subs(t_abs, S)) == 0

# the shifted pieces are Gamma(s + T)
assert simplify(ramp_neg_st - ramp_neg_s.subs(s, s + T)) == 0
assert simplify(plateau_st - plateau_s) == 0
assert simplify(ramp_pos_st - ramp_pos_s.subs(s, s + T)) == 0
print("Gamma(s)/alpha_max^2 matches eq:Gamma_piecewise on all three regions  [verified]")

## Interval 1: $0 \leq \tau \leq \tau_s$

Sorted breakpoints in $[-S, S-T]$: $-S,\, -L-T,\, -L,\, L-T,\, L,\, S-T$

Five sub-intervals with $(\Gamma(s),\, \Gamma(s+T))$ regions:
1. $[-S, -L-T]$: ramp $\times$ ramp
2. $[-L-T, -L]$: ramp $\times$ plateau
3. $[-L, L-T]$: plateau $\times$ plateau
4. $[L-T, L]$: plateau $\times$ ramp
5. $[L, S-T]$: ramp $\times$ ramp

In [ ]:
I1 = simplify(integrate(ramp_neg_s * ramp_neg_st, (s, -S, -L - T)))
I2 = simplify(integrate(ramp_neg_s * plateau_st, (s, -L - T, -L)))
I3 = simplify(integrate(plateau_s * plateau_st, (s, -L, L - T)))
I4 = simplify(integrate(plateau_s * ramp_pos_st, (s, L - T, L)))
I5 = simplify(integrate(ramp_pos_s * ramp_pos_st, (s, L, S - T)))

R1 = simplify(I1 + I2 + I3 + I4 + I5)
print("R_Gamma for T in [0, tau_s]:")
display(R1)

In [ ]:
# Collect as polynomial in T
R1_poly = Poly(sp.expand(R1), T)
print("Polynomial coefficients (T^k):")
for power, coeff in sorted(R1_poly.as_dict().items(), reverse=True):
    print(f"  T^{power[0]}: {simplify(coeff)}")

# Verify R_Gamma(0) = amax^4 * (2L + 2*ts/5)
R_at_0 = R1.subs(T, 0)
expected_R0 = a2**2 * (2 * L + 2 * ts / 5)
assert simplify(R_at_0 - expected_R0) == 0
print(f"\nR_Gamma(0) = alpha_max^4 * (2L + 2*tau_s/5)  [verified]")

## Interval 2: $\tau_s \leq \tau \leq \ell$

For $T > \tau_s$: $-L-T < -S$, so the ramp-ramp overlap on the left vanishes.
Also $S-T < L$, so the ramp-ramp overlap on the right vanishes.

Three sub-intervals: $[-S, -L]$, $[-L, L-T]$, $[L-T, S-T]$

In [ ]:
J1 = simplify(integrate(ramp_neg_s * plateau_st, (s, -S, -L)))
J2 = simplify(integrate(plateau_s * plateau_st, (s, -L, L - T)))
J3 = simplify(integrate(plateau_s * ramp_pos_st, (s, L - T, S - T)))

R2 = simplify(J1 + J2 + J3)
print("R_Gamma for T in [tau_s, 2L]:")
display(R2)
print("\nRemarkably linear!")

## Interval 3: $\ell \leq \tau \leq \ell + \tau_s$

The plateau-plateau overlap vanishes. Breakpoints: $-S,\, L-T,\, -L,\, S-T$
(with $L-T < -L$ since $T > 2L$).

Three sub-intervals: $[-S, L-T]$, $[L-T, -L]$, $[-L, S-T]$

In [ ]:
K1 = simplify(integrate(ramp_neg_s * plateau_st, (s, -S, L - T)))
K2 = simplify(integrate(ramp_neg_s * ramp_pos_st, (s, L - T, -L)))
K3 = simplify(integrate(plateau_s * ramp_pos_st, (s, -L, S - T)))

R3 = simplify(K1 + K2 + K3)
print("R_Gamma for T in [2L, 2L+tau_s]:")
display(R3)

## Interval 4: $\ell + \tau_s \leq \tau \leq \ell + 2\tau_s$

Only ramp-ramp overlap remains: $L-T < -S$, so the only sub-interval is
$[-S, S-T]$: ramp (negative) $\times$ ramp (positive).

In [ ]:
M1 = simplify(integrate(ramp_neg_s * ramp_pos_st, (s, -S, S - T)))

R4 = M1
print("R_Gamma for T in [2L+tau_s, 2S]:")
display(R4)

# Show that it factors as (2S - T)^5 / (30*ts^4)
R4_factored = simplify(R4 / (a2**2))
print("\nR4 / alpha_max^4 =")
display(R4_factored)
assert simplify(R4 - a2**2 * (2 * S - T)**5 / (30 * ts**4)) == 0
print("= (2S - T)^5 / (30 * tau_s^4)  [verified]")

## Convert to $\ell$, $\tau_s$ variables

Substitute $L = \ell/2$ to express intervals in terms of the physical parameters.

In [ ]:
ell_sym = sp.Symbol('ell', positive=True)
tau_s_sym = sp.Symbol('tau_s', positive=True)
subs_dict = {L: ell_sym / 2, ts: tau_s_sym}

labels = [
    r"0 \leq \tau \leq \tau_s",
    r"\tau_s \leq \tau \leq \ell",
    r"\ell \leq \tau \leq \ell + \tau_s",
    r"\ell + \tau_s \leq \tau \leq \ell + 2\tau_s",
]

R_pieces = []
print("CLOSED-FORM R_Gamma(tau) (Eq. R_Gamma_closed):")
print("=" * 60)
for label, R_expr in zip(labels, [R1, R2, R3, R4]):
    R_sub = simplify(R_expr.subs(subs_dict))
    R_pieces.append(R_sub)
    print(f"\n  {label}:")
    display(Math(rf"R_\Gamma(\tau) = {latex(R_sub)}"))

print(f"\n  tau > ell + 2*tau_s:  R_Gamma = 0")

## Comparison with the paper (Eqs. `eq:R_Gamma_closed` and `eq:P3`)

Eqs. `eq:R_Gamma_closed` and `eq:P3` of the manuscript (Appendix `app:acf_trapezoid`) are taken from the equation registry (`equations.py`), where they are transcribed into SymPy term by term, exactly as typeset and independently of the expressions derived above. `check_equations.py` keeps that transcription in sync with `ms.tex` and checks spotgp's implementation against it. The difference from each derived piece must simplify to zero.

**Normalization convention:** the paper uses the normalized envelope $\Gamma = \alpha^2/\alpha_{\max}^2$, so its $R_\Gamma$ has no $\alpha_{\max}^4$ factor. This notebook keeps $\Gamma = \alpha^2$. We therefore compare $R_\Gamma^{\rm derived}/\alpha_{\max}^4$ with the paper's expression, with $\ell = 2L$ and lag $\tau = T$, and also assert that $\alpha_{\max}$ drops out entirely. The interval breakpoints are compared as well.

In [ ]:
# Eqs. (R_Gamma_closed) and (P3) from the equation registry, in paper notation:
# lag tau, plateau ell (= 2L), ramp tau_s. Normalized Gamma => no alpha_max^4.
import equations as eqs

tau_p, ell_p, tsp = eqs.tau, eqs.ell, eqs.tau_s
*branches, (paper_R5, otherwise) = eqs.get("eq:R_Gamma_closed").expr.args  # zero beyond the last
assert otherwise == True
paper_R1, paper_R2, paper_R3, paper_R4 = [piece for piece, _ in branches]
assert paper_R3 == eqs.get("eq:P3").expr                                    # the third piece is P_3
# each branch holds for tau <= its upper end (SymPy may store tau <= b as b >= tau)
ends = [cond if cond.lhs == tau_p else cond.reversed for _, cond in branches]
assert all(isinstance(c, sp.LessThan) and c.lhs == tau_p for c in ends), ends
paper_bounds = [sp.Integer(0)] + [c.rhs for c in ends]

paper_pieces = [paper_R1, paper_R2, paper_R3, paper_R4]

In [ ]:
# Interval breakpoints: derived [0, ts, 2L, 2L+ts, 2S] with L = ell/2
derived_bounds = [sp.Integer(0), ts, 2 * L, 2 * L + ts, 2 * S]
for b_der, b_pap in zip(derived_bounds, paper_bounds):
    assert simplify(b_der.subs(subs_dict) - b_pap) == 0, (b_der, b_pap)

print("Derived R_Gamma / alpha_max^4  vs  eq:R_Gamma_closed (paper):")
for k, (R_sub, paper_piece) in enumerate(zip(R_pieces, paper_pieces), start=1):
    derived_piece = sp.simplify(R_sub.subs(T, tau_p) / amax**4)
    assert amax not in derived_piece.free_symbols  # R_Gamma scales exactly as alpha_max^4
    diff = sp.simplify(derived_piece - paper_piece)
    print(f"  piece {k} ({labels[k - 1]}):  derived - paper = {diff}")
    assert diff == 0, f"piece {k} of eq:R_Gamma_closed disagrees with the derivation: {diff}"
print("All four pieces of eq:R_Gamma_closed (incl. P_3, eq:P3) match the derivation  [verified]")

## Numerical verification

Brute-force check: compute $R_\Gamma(\tau)$ by trapezoid-rule integration of $\Gamma(s)\,\Gamma(s+\tau)$ on a fine grid, at one lag inside each interval of Eq. `eq:R_Gamma_closed` plus one lag beyond the support (where $R_\Gamma = 0$). Both the derived pieces and the paper's transcribed pieces (rescaled by $\alpha_{\max}^4$) must agree with it.

**Tolerance:** the original absolute criterion $|R_{\rm sym} - R_{\rm num}| < 10^{-8}$ is kept, and a relative criterion of $10^{-6}$ is added. With grid spacing $h \approx 10^{-3}$, the trapezoid-rule error (including the kinks of $\Gamma$ at $|s| = \ell/2$) is $\lesssim h^2 \sim 10^{-6}$ in relative terms, and observed errors are $\lesssim 10^{-8}$. Because every $R_\Gamma$ value tested here is $< 10^{-2}$, the relative criterion is the stricter of the two.

In [ ]:
import numpy as np

def Gamma_numpy(s_arr, L_n, ts_n, amax_n):
    abs_s = np.abs(s_arr)
    S_n = L_n + ts_n
    result = np.zeros_like(abs_s)
    plateau = abs_s <= L_n
    ramp = (abs_s > L_n) & (abs_s <= S_n)
    result[plateau] = amax_n**2
    result[ramp] = amax_n**2 * ((S_n - abs_s[ramp]) / ts_n)**2
    return result

def R_Gamma_numerical(tau_arr, L_n, ts_n, amax_n, ns=20000):
    S_n = L_n + ts_n
    s_grid = np.linspace(-S_n - 0.01, S_n + 0.01, ns)
    G_s = Gamma_numpy(s_grid, L_n, ts_n, amax_n)
    R = np.empty_like(tau_arr)
    for i, t in enumerate(tau_arr):
        G_st = Gamma_numpy(s_grid + t, L_n, ts_n, amax_n)
        R[i] = np.trapezoid(G_s * G_st, s_grid)
    return R

In [ ]:
# Test with L=7.5, ts=3, amax=0.1 (i.e., ell=15, tau_s=3)
L_n, ts_n, amax_n = 7.5, 3.0, 0.1

ABS_TOL = 1e-8  # original absolute criterion (R values here are ~1e-7 to 2e-3)
REL_TOL = 1e-6  # trapezoid rule with h ~ 1e-3: error <~ h^2 ~ 1e-6 relative; observed <~ 1e-8

test_points = [
    (1.5,  "[0, tau_s]",              R1,            paper_R1),
    (7.0,  "[tau_s, ell]",            R2,            paper_R2),
    (16.5, "[ell, ell+tau_s]",        R3,            paper_R3),
    (19.5, "[ell+tau_s, ell+2tau_s]", R4,            paper_R4),
    (22.0, "tau > ell+2tau_s",        sp.Integer(0), paper_R5),
]

def close_to_numerical(R_val, R_num):
    err = abs(R_val - R_num)
    return err < ABS_TOL and err <= REL_TOL * abs(R_num)

print("Numerical verification:")
print("=" * 60)
for T_val, label, R_expr, paper_expr in test_points:
    R_sym = float(R_expr.subs([(T, T_val), (L, L_n), (ts, ts_n), (amax, amax_n)]))
    # paper's normalized R_Gamma, rescaled by alpha_max^4 to this notebook's convention
    R_pap = float(paper_expr.subs([(tau_p, T_val), (ell_p, 2 * L_n), (tsp, ts_n)])) * amax_n**4
    R_num = R_Gamma_numerical(np.array([T_val]), L_n, ts_n, amax_n)[0]
    match = close_to_numerical(R_sym, R_num)
    match_paper = close_to_numerical(R_pap, R_num)
    rel_err = abs(R_sym - R_num) / max(abs(R_num), 1e-300)
    print(f"  T={T_val:5.1f} {label:30s}: sym={R_sym:.10e}, num={R_num:.10e}, "
          f"rel_err={rel_err:.1e}, match={match}, paper_match={match_paper}")
    assert match, f"derived R_Gamma disagrees with numerical integral at T={T_val} ({label})"
    assert match_paper, f"eq:R_Gamma_closed disagrees with numerical integral at T={T_val} ({label})"

## Validity range: $\ell \geq \tau_s$

The symbolic derivation above assumes $\ell > 2\tau_s$, but Appendix `app:acf_trapezoid` states the closed form for all $\ell \geq \tau_s$ (plateau at least as long as one ramp), which is also the range `spotgp` accepts. For $\tau_s \leq \ell < 2\tau_s$ the sub-intervals of the convolution are arranged differently, so here the paper's Eq. `eq:R_Gamma_closed` (from the equation registry) is compared with the autocorrelation integral $\int \Gamma(s)\,\Gamma(s+\tau)\,ds$ of the normalized envelope, computed piece by piece at 30 digits (the integrand is a polynomial between the breakpoints, so the quadrature is exact to working precision). Lags run across all four intervals and past the support. For $\ell < \tau_s$ the closed form must fail, which shows that $\ell \geq \tau_s$ is the right bound.

In [ ]:
import mpmath
import equations as eqs

mpmath.mp.dps = 30
R_paper = sp.lambdify((eqs.tau, eqs.ell, eqs.tau_s), eqs.get("eq:R_Gamma_closed").expr, "mpmath")


def Gamma_norm(s, ell_v, ts_v):
    # Eq. Gamma_piecewise (normalized, unit plateau)
    a = abs(s)
    if a <= ell_v / 2:
        return mpmath.mpf(1)
    if a <= ell_v / 2 + ts_v:
        return ((ell_v / 2 + ts_v - a) / ts_v)**2
    return mpmath.mpf(0)


def R_integral(lag, ell_v, ts_v):
    # int Gamma(s) Gamma(s + lag) ds, split at every breakpoint of both factors
    S_v = ell_v / 2 + ts_v
    pts = sorted({-S_v, -ell_v / 2, ell_v / 2, S_v, -S_v - lag, -ell_v / 2 - lag, ell_v / 2 - lag, S_v - lag})
    pts = [x for x in pts if -S_v <= x <= S_v - lag]
    if len(pts) < 2:
        return mpmath.mpf(0)
    return mpmath.quad(lambda s: Gamma_norm(s, ell_v, ts_v) * Gamma_norm(s + lag, ell_v, ts_v), pts)


def max_error(ell_v, ts_v, n_lags=41):
    ell_v, ts_v = mpmath.mpf(ell_v), mpmath.mpf(ts_v)
    lags = [ (ell_v + 2 * ts_v + 1) * k / (n_lags - 1) for k in range(n_lags) ]
    return max(abs(R_paper(l, ell_v, ts_v) - R_integral(l, ell_v, ts_v)) for l in lags)


print("max |eq:R_Gamma_closed - integral| over lags in [0, ell + 2 tau_s + 1]:")
for ell_v, ts_v in [("15", "3"), ("5", "3"), ("3.2", "3"), ("4", "3.9"), ("3", "3")]:   # ell >= tau_s
    err = max_error(ell_v, ts_v)
    print(f"  ell = {ell_v:>4}, tau_s = {ts_v:>3}: {mpmath.nstr(err, 3)}")
    assert err < 1e-20, f"eq:R_Gamma_closed fails at ell = {ell_v}, tau_s = {ts_v}"
for ell_v, ts_v in [("2.5", "3"), ("1", "3")]:                                           # ell < tau_s
    err = max_error(ell_v, ts_v)
    print(f"  ell = {ell_v:>4}, tau_s = {ts_v:>3}: {mpmath.nstr(err, 3)}   (ell < tau_s: closed form must fail)")
    assert err > 1e-4
print("Verified: eq:R_Gamma_closed holds for ell >= tau_s, including tau_s <= ell < 2 tau_s, and fails for ell < tau_s")

## Continuity at the interval boundaries

Adjacent pieces must agree at $\tau = \tau_s$, $\ell$ and $\ell + \tau_s$, and the last piece must vanish at $\tau = \ell + 2\tau_s$, the end of the support (Eq. `eq:R_Gamma_closed`).

In [ ]:
# Continuity check at interval boundaries
print("Continuity at boundaries:")

# At T = tau_s
R1_at_ts = simplify(R1.subs(T, ts))
R2_at_ts = simplify(R2.subs(T, ts))
print(f"  T=tau_s: R1={R1_at_ts}, R2={R2_at_ts}, match={simplify(R1_at_ts - R2_at_ts) == 0}")
assert simplify(R1_at_ts - R2_at_ts) == 0, "discontinuity at T = tau_s"

# At T = 2L (= ell)
R2_at_2L = simplify(R2.subs(T, 2 * L))
R3_at_2L = simplify(R3.subs(T, 2 * L))
print(f"  T=2L:    R2={R2_at_2L}, R3={R3_at_2L}, match={simplify(R2_at_2L - R3_at_2L) == 0}")
assert simplify(R2_at_2L - R3_at_2L) == 0, "discontinuity at T = ell"

# At T = 2L + ts (= ell + tau_s)
R3_at_2Lts = simplify(R3.subs(T, 2 * L + ts))
R4_at_2Lts = simplify(R4.subs(T, 2 * L + ts))
print(f"  T=2L+ts: R3={R3_at_2Lts}, R4={R4_at_2Lts}, match={simplify(R3_at_2Lts - R4_at_2Lts) == 0}")
assert simplify(R3_at_2Lts - R4_at_2Lts) == 0, "discontinuity at T = ell + tau_s"

# At T = 2S (= ell + 2*tau_s): should be zero
R4_at_2S = simplify(R4.subs(T, 2 * S))
print(f"  T=2S:    R4={R4_at_2S}  (should be 0)")
assert R4_at_2S == 0, "R_Gamma does not vanish at T = ell + 2 tau_s"

## Physical interpretation

- **Interval 1** ($0 \leq \tau \leq \tau_s$): Degree-5 polynomial from all overlap types. No linear term, so $R_\Gamma'(0) = 0$.
- **Interval 2** ($\tau_s \leq \tau \leq \ell$): Remarkably **linear**: $R_\Gamma = \alpha_{\max}^4(\ell + 2\tau_s/3 - \tau)$. The slope $-\alpha_{\max}^4$ reflects the constant rate at which the plateau-plateau overlap shrinks.
- **Interval 3** ($\ell \leq \tau \leq \ell + \tau_s$): Degree-5 polynomial $P_3(\tau)$.
- **Interval 4** ($\ell + \tau_s \leq \tau \leq \ell + 2\tau_s$): Factors as $\alpha_{\max}^4(\ell + 2\tau_s - \tau)^5/(30\tau_s^4)$, vanishing smoothly at $\tau = \ell + 2\tau_s$.